# Council of LLMs: what the models never see on screen

Three models walk into a group chat. Two argue, one judges, and none of them remembers a single word from one turn to the next.

This notebook opens the envelope each model receives, so you can see how the debate keeps going anyway. No API calls here: everything runs offline.

```
   [ Debater 1 ]        [ Debater 2 ]
         \                  /
          \                /
           [   Judge    ]
                |
           "LOSER: ..."
```

## 1. The scene is built from 4 boxes per seat

Every seat on the page has a model, a name, a purpose and a rule. `personas.py` pours those values into a template: the same sentence with new blanks each time.

In [ ]:
import sys
sys.path.insert(0, "council_app")  # so we can import the app's helper files

from personas import MODELS, debater_prompt, judge_prompt

print(MODELS)

In [ ]:
# fill the blanks for one debater, exactly as app.py does when you press Submit
sukanya = debater_prompt("Sukanya", "prove that Thai food is the best, with evidence", "if you are proven wrong, give up")
print(sukanya)

## 2. The judge gets everyone's rules

A debater only knows its own rule. The judge needs both, otherwise it would be refereeing a match without knowing what a goal is.

In [ ]:
judge = judge_prompt("Judy", "fair but funny", "facts and stats win, not shouting",
                     "Sukanya", "if you are proven wrong, give up",
                     "Kumar", "if you are proven wrong, give up")
print(judge)

## 3. The illusion of memory

The page shows a growing conversation. The models see none of it unless we send it. So on **every** turn, `ask()` sends a fresh envelope:

| Part of the envelope | Shown on the page? |
|---|---|
| system: that seat's filled template | No |
| user: the whole transcript so far | Yes |
| user tail: "You are X. Write only your next message." | No |

Below is the envelope Kumar would get on his first turn, built the same way `app.py` builds it, minus the actual API call.

In [ ]:
transcript = "**Judy:** Welcome! Today's debate: is Thai food or Indian food better?\n\n"
transcript += "**Sukanya:** Pad thai. Mango sticky rice. Case closed.\n\n"

kumar = debater_prompt("Kumar", "prove Indian food is the best, with evidence", "if you are proven wrong, give up")

envelope = [
    {"role": "system", "content": kumar},
    {"role": "user", "content": transcript + "\nYou are Kumar. Write only your next message."},
]
for message in envelope:
    print(f"--- {message['role']} ---")
    print(message["content"])

## 4. Things the council taught me

- **Small judges are trigger-happy.** With Llama 3.2 in the judge seat, a loser was named after one round, sometimes right after the judge asked a question. Different loser every run.
- **Big judges want receipts.** With GPT judging, it caught made-up facts ("5 UNESCO cuisines", a "2011 World Food Festival") and refused to crown anyone without evidence. The debaters then made peace and planned a food tour.
- **The name echo.** Because the transcript is full of `**Name:**` lines, models sometimes reply with `Kumar: Kumar: ...`. They copy the pattern they are shown.
- **Next step:** judge by evidence, not opinion. With tool calling (Week 2 Day 4) the judge could run two code answers, time them, and let the stopwatch decide.